# 🔒 Notebook 1: Understanding Race Conditions

Race conditions are one of the most insidious bugs in software. They work perfectly in testing, then fail randomly in production when two requests happen to arrive at the same time.

## Learning Objectives

By the end of this notebook, you'll understand:
- What race conditions are and why they happen
- How to reproduce and visualize race conditions
- Why basic code fails under concurrency
- The "check-then-act" anti-pattern

## 🎫 The Concert Ticket Problem

Imagine you're building Ticketmaster. There's 1 seat left for The Weeknd concert. Alice and Bob both click "Buy Now" at the same moment.

```
The naive approach:

def buy_ticket(user_id, concert_id):
    # Step 1: Check availability
    seats = get_available_seats(concert_id)
    
    # Step 2: Decide
    if seats >= 1:
        # Step 3: Update
        decrement_seats(concert_id)
        charge_user(user_id)
        return "Success!"
    else:
        return "Sold out!"
```

What could go wrong? Everything.

## 🛠️ Let's Set Up

First, start PostgreSQL and visualization tools:

```bash
cd patterns/dealing-with-contention
docker-compose up -d
```

### 🔍 Open Adminer to Watch Database Changes

While running the notebooks, keep **Adminer** open in a browser tab:

1. Go to **http://localhost:8080**
2. Login with:
   - System: `PostgreSQL`
   - Server: `postgres`
   - Username: `demo`
   - Password: `demo`
   - Database: `contention_demo`
3. Click on `concerts` table → "Select data" to see available seats
4. Click "Refresh" after running cells to see changes!

💡 **Tip**: Watch the `available_seats` column as you run the race condition demos!

In [ ]:
import psycopg2
import time
from concurrent.futures import ThreadPoolExecutor
from threading import Lock

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "contention_demo",
    "user": "demo",
    "password": "demo"
}

def get_connection():
    return psycopg2.connect(**DB_CONFIG)

try:
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT version()")
    version = cursor.fetchone()[0]
    print(f"✅ Connected to PostgreSQL")
    print(f"   Version: {version.split(',')[0]}")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Make sure PostgreSQL is running: docker-compose up -d")

In [ ]:
def check_concert_seats():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT name, available_seats, price FROM concerts")
    concerts = cursor.fetchall()
    conn.close()
    
    print("🎤 Available Concerts")
    print("=" * 60)
    for name, seats, price in concerts:
        print(f"   {name}")
        print(f"   Seats: {seats} | Price: ${price}")
        print()

check_concert_seats()

## 🐛 The Naive Implementation (Broken!)

Let's implement the "check-then-act" pattern that most beginners write. This code looks correct but has a fatal flaw.

In [ ]:
results_log = []
log_lock = Lock()

def naive_buy_ticket(user_id: str, concert_id: int = 1):
    conn = get_connection()
    cursor = conn.cursor()
    
    try:
        cursor.execute(
            "SELECT available_seats FROM concerts WHERE id = %s",
            (concert_id,)
        )
        seats = cursor.fetchone()[0]
        
        with log_lock:
            results_log.append(f"{user_id} reads: {seats} seats available")
        
        time.sleep(0.01)
        
        if seats >= 1:
            cursor.execute(
                "UPDATE concerts SET available_seats = available_seats - 1 WHERE id = %s",
                (concert_id,)
            )
            conn.commit()
            
            with log_lock:
                results_log.append(f"{user_id} BOUGHT a ticket! ✅")
            return True
        else:
            with log_lock:
                results_log.append(f"{user_id} sees SOLD OUT ❌")
            return False
            
    finally:
        conn.close()

print("✅ Naive buy_ticket function created")
print("   This implementation has a race condition!")

In [ ]:
def reset_concert_seats(concert_id: int = 1, seats: int = 1):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE concerts SET available_seats = %s WHERE id = %s",
        (seats, concert_id)
    )
    conn.commit()
    conn.close()

reset_concert_seats(1, 1)
print("🔄 Reset: The Weeknd concert now has exactly 1 seat")

## 💥 Reproducing the Race Condition

Now let's simulate Alice and Bob trying to buy the last ticket at the same time.

In [ ]:
reset_concert_seats(1, 1)
results_log.clear()

print("🎫 Race Condition Demonstration")
print("=" * 50)
print("Starting state: 1 seat available")
print("Alice and Bob both click 'Buy Now'...")
print()

with ThreadPoolExecutor(max_workers=2) as executor:
    future_alice = executor.submit(naive_buy_ticket, "Alice")
    future_bob = executor.submit(naive_buy_ticket, "Bob")
    
    alice_result = future_alice.result()
    bob_result = future_bob.result()

print("📋 Event Log:")
for event in results_log:
    print(f"   {event}")

print()
print(f"Alice got ticket: {alice_result}")
print(f"Bob got ticket: {bob_result}")

conn = get_connection()
cursor = conn.cursor()
cursor.execute("SELECT available_seats FROM concerts WHERE id = 1")
final_seats = cursor.fetchone()[0]
conn.close()

print(f"\n📊 Final seat count: {final_seats}")

if alice_result and bob_result:
    print("\n🔥 RACE CONDITION OCCURRED!")
    print("   Both Alice and Bob bought the same seat!")
    print("   Someone is getting kicked out of the concert...")
elif final_seats < 0:
    print("\n🔥 OVERSOLD! Seats went negative!")

## 📊 Run It Multiple Times

Race conditions are probabilistic. Let's run this many times to see how often it fails.

In [ ]:
def run_race_condition_test():
    reset_concert_seats(1, 1)
    
    with ThreadPoolExecutor(max_workers=2) as executor:
        future_alice = executor.submit(naive_buy_ticket, "Alice")
        future_bob = executor.submit(naive_buy_ticket, "Bob")
        
        alice = future_alice.result()
        bob = future_bob.result()
    
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT available_seats FROM concerts WHERE id = 1")
    final_seats = cursor.fetchone()[0]
    conn.close()
    
    return {
        "both_bought": alice and bob,
        "oversold": final_seats < 0,
        "correct": (alice != bob) and final_seats == 0
    }

print("🧪 Running 20 race condition tests...")
print()

results = {"both_bought": 0, "oversold": 0, "correct": 0}

for i in range(20):
    results_log.clear()
    result = run_race_condition_test()
    
    if result["both_bought"]:
        results["both_bought"] += 1
        status = "🔥 DOUBLE SOLD"
    elif result["oversold"]:
        results["oversold"] += 1
        status = "🔥 OVERSOLD"
    else:
        results["correct"] += 1
        status = "✅ Correct"
    
    print(f"   Test {i+1:2d}: {status}")

print()
print("📊 Results Summary")
print("=" * 40)
print(f"   Correct outcomes:  {results['correct']}/20 ({results['correct']*5}%)")
print(f"   Double sold:       {results['both_bought']}/20 ({results['both_bought']*5}%)")
print(f"   Oversold:          {results['oversold']}/20 ({results['oversold']*5}%)")
print()
if results["correct"] < 20:
    print("💡 Even with just 2 concurrent users, race conditions occur!")
    print("   Imagine 10,000 users hitting 'Buy' at ticket drop time...")

## 🔍 Why Does This Happen?

The race condition occurs because of the **check-then-act** pattern:

```
Timeline:
────────────────────────────────────────────────────────────────
Time     Alice                        Bob
────────────────────────────────────────────────────────────────
T1       SELECT seats (reads 1)       
T2                                    SELECT seats (reads 1)
T3       if 1 >= 1 → true            
T4                                    if 1 >= 1 → true
T5       UPDATE seats = seats - 1    
T6                                    UPDATE seats = seats - 1
T7       COMMIT (seats now = 0)      
T8                                    COMMIT (seats now = -1!)
────────────────────────────────────────────────────────────────
```

**The fundamental problem:** There's a gap between reading and writing where the world can change.

## 🎯 The Check-Then-Act Anti-Pattern

This pattern appears everywhere and is almost always wrong under concurrency:

```python
# Anti-pattern 1: Inventory check
if get_inventory(item) >= quantity:
    reduce_inventory(item, quantity)  # ❌ Race condition!

# Anti-pattern 2: Balance check
if get_balance(account) >= amount:
    debit_account(account, amount)  # ❌ Race condition!

# Anti-pattern 3: Seat availability
if seat_is_available(seat_id):
    reserve_seat(seat_id)  # ❌ Race condition!

# Anti-pattern 4: Username check
if not username_exists(username):
    create_user(username)  # ❌ Race condition!
```

Any time you read state, make a decision, then write based on that decision, you have a potential race condition.

## 📈 The Scale Problem

Let's see what happens with more concurrent users.

In [ ]:
def stress_test(num_users: int, available_seats: int):
    reset_concert_seats(1, available_seats)
    
    successful_purchases = 0
    
    def try_buy(user_num):
        return naive_buy_ticket(f"User{user_num}")
    
    with ThreadPoolExecutor(max_workers=num_users) as executor:
        futures = [executor.submit(try_buy, i) for i in range(num_users)]
        results = [f.result() for f in futures]
        successful_purchases = sum(results)
    
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute("SELECT available_seats FROM concerts WHERE id = 1")
    final_seats = cursor.fetchone()[0]
    conn.close()
    
    return {
        "users": num_users,
        "seats": available_seats,
        "purchases": successful_purchases,
        "final_seats": final_seats,
        "oversold": successful_purchases > available_seats
    }

print("🧪 Stress Testing Race Conditions")
print("=" * 50)
print()

results_log.clear()

test_cases = [
    (5, 3),
    (10, 5),
    (20, 10),
    (50, 20),
]

for num_users, seats in test_cases:
    result = stress_test(num_users, seats)
    status = "🔥 OVERSOLD!" if result["oversold"] else "✅"
    print(f"Users: {result['users']:3d} | Seats: {result['seats']:3d} | "
          f"Sold: {result['purchases']:3d} | Final: {result['final_seats']:3d} | {status}")

print()
print("💡 More concurrent users = More race conditions!")
print("   This is why we need proper synchronization.")

## 🧪 Quick Quiz

1. **Why does the naive implementation fail?**

2. **Would adding a simple `if seats >= 1` check in the database fix it?**

3. **Does this problem get better or worse as you scale?**

In [ ]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. The naive implementation fails because there's a GAP between")
print("   reading the seat count and updating it. In that gap, another")
print("   transaction can read the same value.")
print()
print("2. NO! A simple WHERE clause doesn't help because:")
print("   - Both transactions read seats=1 BEFORE either writes")
print("   - Both UPDATE statements would execute with seats >= 1")
print("   - We need to make read+check+write ATOMIC")
print()
print("3. WORSE! More concurrent users means:")
print("   - Higher probability of overlapping operations")
print("   - More conflicts on the same resources")
print("   - Race condition windows become more likely to be hit")

## 📚 Summary

### What We Learned

1. **Race conditions** occur when multiple processes compete for the same resource
2. **Check-then-act** is an anti-pattern that creates race condition windows
3. **The gap** between read and write is where chaos happens
4. **Scaling** makes race conditions worse, not better
5. **Testing** may not catch race conditions - they're probabilistic

### The Core Problem

> Reading state, making a decision, and writing based on that decision are **three separate operations**. We need to make them **atomic** (all-or-nothing) to prevent race conditions.

### Next Up: Transactions

In the next notebook, we'll learn how **database transactions** provide atomicity - and why they alone don't solve all race conditions!